In [1]:
!pip install pyspark

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DataPipeline") \
    .getOrCreate()

spark

### 📁 Estrutura de Diretórios

Definição da estrutura base de diretórios para organização do pipeline de dados no ambiente.

As pastas são separadas por camadas lógicas:

* **raw**: dados de entrada
* **trusted**: dados tratados e padronizados
* **refined**: dados prontos para consumo analítico

Boas práticas:

* Utilizar caminhos padronizados e reutilizáveis
* Garantir isolamento entre camadas
* Criar diretórios de forma idempotente (`exist_ok=True`)
* Facilitar manutenção e escalabilidade do pipeline

### FAÇA O UPLOAD DOS 3 ARQUIVOS DE ORIGEM NA CAMADA RAW


In [2]:
import os

base_path = "/content/data"

raw_path = f"{base_path}/raw"
trusted_path = f"{base_path}/trusted"
refined_path = f"{base_path}/refined"

os.makedirs(raw_path, exist_ok=True)
os.makedirs(trusted_path, exist_ok=True)
os.makedirs(refined_path, exist_ok=True)

print("Pastas criadas:")
print(raw_path)
print(trusted_path)
print(refined_path)

Pastas criadas:
/content/data/raw
/content/data/trusted
/content/data/refined


### 📌 Camadas RAW → TRUSTED (Customers)

Nesta etapa do pipeline, aplicamos o conceito de arquitetura em camadas para garantir qualidade, rastreabilidade e organização dos dados.

#### 🔹 Camada RAW

A camada **raw** representa os dados exatamente como foram ingeridos da fonte, sem qualquer tipo de transformação.
Aqui, o objetivo é preservar o dado original para auditoria e reprocessamento futuro.

No caso de `customers`, os dados são lidos diretamente do arquivo CSV, mantendo o schema inferido automaticamente pelo Spark:

* Nenhum tratamento é aplicado
* Possíveis inconsistências ainda existem (nulos, duplicidades, formatação irregular)
* Serve como fonte confiável de reprocessamento

---

#### 🔹 Camada TRUSTED

A camada **trusted** tem como objetivo transformar os dados brutos em um formato confiável e padronizado, garantindo qualidade mínima para consumo analítico.

Para isso, foram aplicadas as seguintes regras:

* **Tipagem de dados:** conversão de `customer_id` para inteiro e `created_at` para timestamp
* **Remoção de duplicidades:** considerando `customer_id` como chave única
* **Tratamento de nulos críticos:** remoção de registros sem `name`
* **Padronização de texto:**

  * `name` com espaços removidos
  * `email` em lowercase
  * `country` em uppercase
* **Auditoria:** criação da coluna `ingestion_timestamp` com o horário de processamento

Essa camada garante que os dados estejam:

* Consistentes
* Padronizados
* Prontos para transformação em camadas analíticas (refined/gold)

---

#### 💡 Observação

A separação entre **raw** e **trusted** permite reprocessamento seguro e rastreabilidade, além de evitar que erros na transformação afetem a integridade dos dados originais.


In [3]:
df_customers = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(f"{raw_path}/customers.csv")

df_customers.show()
df_customers.printSchema()

+-----------+--------------+--------------------+-------+----------+
|customer_id|          name|               email|country|created_at|
+-----------+--------------+--------------------+-------+----------+
|          1|     Ana Silva|       ana@email.com|     BR|2022-01-10|
|          2|      John Doe|      john@email.com|     US|2021-11-03|
|          3|   Maria Gomez|     maria@email.com|     AR|2023-02-20|
|          3|   Maria Gomez|     maria@email.com|     AR|2023-02-20|
|          4|Carlos Pereira|    carlos@email.com|     BR|2023-05-15|
|          5|          NULL|empty_name@email.com|     BR|2023-06-01|
|          6|   Julia Smith|     julia@email.com|   NULL|2023-06-10|
+-----------+--------------+--------------------+-------+----------+

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- created_at: date (nullable = true)



In [13]:
from pyspark.sql.functions import col, to_timestamp, current_timestamp, lower, trim, upper

df_customers_trusted = df_customers \
    .withColumn("customer_id", col("customer_id").cast("int")) \
    .withColumn("created_at", to_timestamp("created_at")) \
    .dropDuplicates(["customer_id"]) \
    .dropna(subset=["name"]) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("name", trim(col("name"))) \
    .withColumn("email", lower(trim(col("email")))) \
    .withColumn("country", upper(trim(col("country"))))

df_customers_trusted.write \
    .mode("overwrite") \
    .parquet(f"{trusted_path}/tru_customers")

df_customers_trusted.show()
df_customers_trusted.printSchema()

+-----------+--------------+----------------+-------+-------------------+--------------------+
|customer_id|          name|           email|country|         created_at| ingestion_timestamp|
+-----------+--------------+----------------+-------+-------------------+--------------------+
|          1|     Ana Silva|   ana@email.com|     BR|2022-01-10 00:00:00|2026-04-15 23:36:...|
|          2|      John Doe|  john@email.com|     US|2021-11-03 00:00:00|2026-04-15 23:36:...|
|          3|   Maria Gomez| maria@email.com|     AR|2023-02-20 00:00:00|2026-04-15 23:36:...|
|          4|Carlos Pereira|carlos@email.com|     BR|2023-05-15 00:00:00|2026-04-15 23:36:...|
|          6|   Julia Smith| julia@email.com|   NULL|2023-06-10 00:00:00|2026-04-15 23:36:...|
+-----------+--------------+----------------+-------+-------------------+--------------------+

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (

### 📌 RAW → TRUSTED (Events)

Aplicação da arquitetura em camadas para processamento de eventos, mantendo separação entre ingestão e padronização.

A camada **raw** preserva os dados no formato original (JSONL), respeitando a estrutura de entrada e garantindo rastreabilidade.

A camada **trusted** aplica transformações para garantir consistência e qualidade dos dados:

* **Tipagem de dados:** conversão de `customer_id` para inteiro e tratamento de `event_timestamp` com abordagem tolerante a erro (`try_cast`), evitando falhas por dados inválidos
* **Padronização de texto:** normalização do campo `event_type` (lower + trim) para reduzir variações e inconsistências
* **Normalização de valores:** agrupamento de variações semânticas de eventos (ex: `log_in` → `login`) para controle de cardinalidade
* **Auditoria:** inclusão da coluna `ingestion_timestamp` para rastreabilidade do processamento
* **Remoção de duplicidades:** garantia de unicidade com base em `event_id`

Boas práticas:

* Padronizar valores categóricos para evitar alta cardinalidade
* Tratar conversões de tipos de forma tolerante a erro
* Garantir unicidade dos eventos
* Incluir metadados de ingestão
* Manter consistência estrutural para alto volume de dados


In [4]:
df_events = spark.read \
    .json(f"{raw_path}/events.jsonl")

df_events.show()
df_events.printSchema()

+-----------+--------+--------------------+----------+
|customer_id|event_id|     event_timestamp|event_type|
+-----------+--------+--------------------+----------+
|          1|      E1|2023-07-10T10:01:00Z|     login|
|          1|      E2|2023-07-10T10:05:00Z|  purchase|
|          2|      E3|2023-07-11T09:00:00Z|     login|
|         99|      E4|2023-07-11T09:10:00Z|     login|
|          3|      E5|2023-07-11T09:15:00Z|    LOG_IN|
|          3|      E6|   invalid_timestamp|     login|
+-----------+--------+--------------------+----------+

root
 |-- customer_id: long (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_timestamp: string (nullable = true)
 |-- event_type: string (nullable = true)



In [19]:
from pyspark.sql.functions import col, to_timestamp, current_timestamp, lower, trim, upper, expr, when

df_events_trusted = df_events \
    .withColumn("customer_id", col("customer_id").cast("int")) \
    .withColumn("event_type", lower(trim(col("event_type")))) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("event_timestamp", expr("try_cast(event_timestamp as timestamp)")) \
    .dropDuplicates(["event_id"]) \
    .withColumn(
        "event_type",
        when(col("event_type").isin("login", "log_in"), "login")
        .otherwise(col("event_type")))

df_events_trusted.write \
    .mode("overwrite") \
    .parquet(f"{trusted_path}/tru_events")

df_events_trusted.show()
df_events_trusted.printSchema()

+-----------+--------+-------------------+----------+--------------------+
|customer_id|event_id|    event_timestamp|event_type| ingestion_timestamp|
+-----------+--------+-------------------+----------+--------------------+
|          1|      E1|2023-07-10 10:01:00|     login|2026-04-15 23:41:...|
|          1|      E2|2023-07-10 10:05:00|  purchase|2026-04-15 23:41:...|
|          2|      E3|2023-07-11 09:00:00|     login|2026-04-15 23:41:...|
|         99|      E4|2023-07-11 09:10:00|     login|2026-04-15 23:41:...|
|          3|      E5|2023-07-11 09:15:00|     login|2026-04-15 23:41:...|
|          3|      E6|               NULL|     login|2026-04-15 23:41:...|
+-----------+--------+-------------------+----------+--------------------+

root
 |-- customer_id: integer (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = false)



### 📌 RAW → TRUSTED (Orders)

Aplicação da arquitetura em camadas para dados de pedidos, garantindo evolução do dado bruto para um formato confiável e padronizado.

Na camada **raw**, os dados são ingeridos diretamente do JSON original, preservando a estrutura de origem e possíveis inconsistências para rastreabilidade e reprocessamento.

Na camada **trusted**, são aplicadas transformações para garantir qualidade e consistência:

* **Tipagem de dados:** conversão de `customer_id` para inteiro e `amount` para double, assegurando compatibilidade com operações analíticas
* **Padronização de texto:** normalização do campo `status` (lower + trim) para evitar variações e inconsistências
* **Tratamento de datas:** conversão de `order_date` considerando múltiplos formatos possíveis, utilizando abordagem tolerante a erro (`try_to_timestamp` + `coalesce`)
* **Auditoria:** inclusão da coluna `ingestion_timestamp` para rastreabilidade do processamento
* **Regra de negócio (refund):** identificação de pedidos com valor negativo através da flag `refund`, preservando informação relevante sem descartar registros
* **Remoção de duplicidades:** garantia de unicidade com base em `order_id`

Essa abordagem mantém o equilíbrio entre qualidade e preservação do dado, evitando perda de informação e preparando a base para consumo em camadas analíticas posteriores.


In [5]:
df_orders = spark.read \
    .option("multiline", True) \
    .json(f"{raw_path}/orders.json")

df_orders.show()
df_orders.printSchema()

+------+--------+-----------+----------+--------+---------+
|amount|currency|customer_id|order_date|order_id|   status|
+------+--------+-----------+----------+--------+---------+
| 250.5|     BRL|          1|2023-07-10|   O1001|completed|
|  99.9|     USD|          2|2023-07-11|   O1002|CANCELLED|
| 180.0|     USD|       NULL|2023-07-12|   O1003|completed|
| -50.0|     BRL|          4|2023-07-13|   O1004|Completed|
| 300.0|     EUR|         99|13-07-2023|   O1005|completed|
+------+--------+-----------+----------+--------+---------+

root
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- status: string (nullable = true)



In [26]:
from pyspark.sql.functions import col, to_timestamp, current_timestamp, lower, trim, upper, expr, when, coalesce

df_orders_trusted = df_orders \
    .withColumn("customer_id", col("customer_id").cast("int")) \
    .withColumn("amount", col("amount").cast("double")) \
    .withColumn("status", lower(trim(col("status")))) \
    .withColumn(
        "order_date",
        coalesce(
            expr("try_to_timestamp(order_date, 'yyyy-MM-dd')"),
            expr("try_to_timestamp(order_date, 'dd-MM-yyyy')")
        )
    ) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("refund", col("amount") < 0) \
    .dropDuplicates(["order_id"])

df_orders_trusted.write \
    .mode("overwrite") \
    .parquet(f"{trusted_path}/tru_orders")

df_orders_trusted.show()
df_orders_trusted.printSchema()

+------+--------+-----------+-------------------+--------+---------+--------------------+------+
|amount|currency|customer_id|         order_date|order_id|   status| ingestion_timestamp|refund|
+------+--------+-----------+-------------------+--------+---------+--------------------+------+
| 250.5|     BRL|          1|2023-07-10 00:00:00|   O1001|completed|2026-04-15 23:58:...| false|
|  99.9|     USD|          2|2023-07-11 00:00:00|   O1002|cancelled|2026-04-15 23:58:...| false|
| 180.0|     USD|       NULL|2023-07-12 00:00:00|   O1003|completed|2026-04-15 23:58:...| false|
| -50.0|     BRL|          4|2023-07-13 00:00:00|   O1004|completed|2026-04-15 23:58:...|  true|
| 300.0|     EUR|         99|2023-07-13 00:00:00|   O1005|completed|2026-04-15 23:58:...| false|
+------+--------+-----------+-------------------+--------+---------+--------------------+------+

root
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- customer_id: integer (nullable = true)

### 📌 Camada Refined e Enriquecimento com API de Moeda

A camada **refined** tem como objetivo disponibilizar dados prontos para consumo analítico, integrando diferentes fontes e aplicando regras de negócio mais avançadas.

Nessa etapa, os dados já tratados na camada trusted são combinados e enriquecidos, garantindo consistência, comparabilidade e maior valor para análise.

Para a base de pedidos, foi realizado um enriquecimento utilizando uma API externa de câmbio, permitindo a padronização dos valores financeiros em uma única moeda (BRL). A integração consiste na obtenção das taxas de conversão, transformação em uma tabela de lookup e posterior junção com os dados de pedidos.

Principais pontos aplicados:

* **Integração com API externa:** consumo de taxas de câmbio atualizadas
* **Transformação de dados:** ajuste da direção da conversão (BRL → outras moedas para outras moedas → BRL)
* **Enriquecimento:** cálculo do valor convertido (`amount_brl`) a partir da moeda original
* **Padronização monetária:** viabiliza comparações e agregações financeiras entre diferentes moedas
* **Precisão e apresentação:** arredondamento dos valores finais para duas casas decimais
* **Persistência:** armazenamento em formato parquet para otimização de leitura e performance

Essa abordagem permite análises financeiras consistentes, independentemente da moeda original dos pedidos, além de demonstrar integração com fontes externas e aplicação de regras de negócio na camada analítica.


In [39]:
import requests

url = "https://v6.exchangerate-api.com/v6/24f0659228f5a949b93b33b8/latest/BRL"

response = requests.get(url)
data = response.json()

rates = data["conversion_rates"]

# transformar em lista (currency, rate)
rates_list = [(k, float(v)) for k, v in rates.items()]

df_rates = spark.createDataFrame(rates_list, ["currency", "rate_brl_to_currency"])

from pyspark.sql.functions import col, round

df_rates = df_rates \
    .withColumn("rate_to_brl", 1 / col("rate_brl_to_currency")) \
    .select("currency", "rate_to_brl")

df_orders_enriched = df_orders_trusted \
    .join(df_rates, "currency", "left") \
    .withColumn("amount_brl", col("amount") * col("rate_to_brl"))

df_orders_enriched = df_orders_enriched \
    .withColumn("amount_brl", round("amount_brl", 2))

df_orders_enriched.write \
    .mode("overwrite") \
    .parquet(f"{refined_path}/ref_orders_brl")

df_orders_enriched.show()
df_orders_enriched.printSchema()

+--------+------+-----------+-------------------+--------+---------+--------------------+------+------------------+----------+
|currency|amount|customer_id|         order_date|order_id|   status| ingestion_timestamp|refund|       rate_to_brl|amount_brl|
+--------+------+-----------+-------------------+--------+---------+--------------------+------+------------------+----------+
|     BRL| 250.5|          1|2023-07-10 00:00:00|   O1001|completed|2026-04-16 00:23:...| false|               1.0|     250.5|
|     USD|  99.9|          2|2023-07-11 00:00:00|   O1002|cancelled|2026-04-16 00:23:...| false|4.9950049950049955|     499.0|
|     USD| 180.0|       NULL|2023-07-12 00:00:00|   O1003|completed|2026-04-16 00:23:...| false|4.9950049950049955|     899.1|
|     BRL| -50.0|          4|2023-07-13 00:00:00|   O1004|completed|2026-04-16 00:23:...|  true|               1.0|     -50.0|
|     EUR| 300.0|         99|2023-07-13 00:00:00|   O1005|completed|2026-04-16 00:23:...| false| 5.889281507656

### 📌 Camada Refined (Orders + Customers)

A camada **refined** representa o nível mais analítico do pipeline, onde os dados são integrados e organizados para consumo direto por análises, dashboards e aplicações de negócio.

Nesta etapa, os dados de pedidos já enriquecidos são combinados com informações de clientes, formando uma visão consolidada e mais rica do domínio.

Principais pontos aplicados:

* **Integração de domínios:** junção entre pedidos e clientes através de `customer_id`, permitindo análises completas de comportamento e receita
* **Seleção estruturada:** definição explícita das colunas relevantes, evitando ambiguidade e garantindo clareza no modelo de dados
* **Padronização de nomenclatura:** diferenciação de metadados de ingestão (`order_ingestion_timestamp` e `customer_ingestion_timestamp`) para manter rastreabilidade
* **Enriquecimento prévio:** utilização de valores já convertidos (`amount_brl`), garantindo consistência em análises financeiras
* **Organização analítica:** estrutura semelhante a uma tabela fato enriquecida, facilitando agregações e consultas

Essa camada consolida os dados em um formato pronto para consumo, reduzindo a complexidade das análises e garantindo maior performance e governança no acesso às informações.


In [64]:
ref_orders_customers = df_orders_enriched \
    .join(df_customers_trusted, "customer_id", "left")

ref_orders_customers = df_orders_enriched.alias("o") \
    .join(df_customers_trusted.alias("c"), "customer_id", "left") \
    .select(
        "customer_id",
        "order_id",
        "amount",
        "amount_brl",
        "currency",
        "status",
        "order_date",
        col("o.ingestion_timestamp").alias("order_ingestion_timestamp"),
        "name",
        "email",
        "country",
        col("c.ingestion_timestamp").alias("customer_ingestion_timestamp")
    )

ref_orders_customers.write \
    .mode("overwrite") \
    .parquet(f"{refined_path}/ref_orders_customers")

ref_orders_customers.show()
ref_orders_customers.printSchema()

+-----------+--------+------+----------+--------+---------+-------------------+-------------------------+--------------+----------------+-------+----------------------------+
|customer_id|order_id|amount|amount_brl|currency|   status|         order_date|order_ingestion_timestamp|          name|           email|country|customer_ingestion_timestamp|
+-----------+--------+------+----------+--------+---------+-------------------+-------------------------+--------------+----------------+-------+----------------------------+
|          1|   O1001| 250.5|     250.5|     BRL|completed|2023-07-10 00:00:00|     2026-04-16 00:50:...|     Ana Silva|   ana@email.com|     BR|        2026-04-16 00:50:...|
|          4|   O1004| -50.0|     -50.0|     BRL|completed|2023-07-13 00:00:00|     2026-04-16 00:50:...|Carlos Pereira|carlos@email.com|     BR|        2026-04-16 00:50:...|
|         99|   O1005| 300.0|   1766.78|     EUR|completed|2023-07-13 00:00:00|     2026-04-16 00:50:...|          NULL|     

### 📌 Camada Refined (Events + Customers)

A camada **refined** consolida os dados em uma visão integrada e orientada a análise, combinando diferentes domínios para facilitar a exploração e geração de insights.

Nesta etapa, os eventos são enriquecidos com informações de clientes, permitindo análises comportamentais mais completas e contextualizadas.

Principais pontos aplicados:

* **Integração de domínios:** junção entre eventos e clientes via `customer_id`, possibilitando associar ações a perfis de usuários
* **Seleção controlada de colunas:** definição explícita dos campos relevantes, garantindo clareza e evitando conflitos de nomenclatura
* **Padronização de metadados:** diferenciação dos timestamps de ingestão (`event_ingestion_timestamp` e `customer_ingestion_timestamp`) para rastreabilidade
* **Contextualização dos eventos:** inclusão de atributos como `name`, `email` e `country`, enriquecendo a análise de comportamento
* **Estrutura analítica:** organização dos dados em formato pronto para consumo, facilitando agregações, segmentações e análises de jornada

Essa camada permite análises mais ricas sobre o comportamento dos usuários, conectando eventos a características dos clientes e reduzindo a complexidade de consultas nas etapas seguintes.


In [62]:
ref_events_customers = df_events_trusted \
    .join(df_customers_trusted, "customer_id", "left") \

ref_events_customers = df_events_trusted.alias("e") \
    .join(df_customers_trusted.alias("c"), "customer_id", "left") \
    .select(
        "customer_id",
        "event_id",
        "event_type",
        "event_timestamp",
        col("e.ingestion_timestamp").alias("event_ingestion_timestamp"),
        col("c.name"),
        col("c.email"),
        col("c.country"),
        col("c.ingestion_timestamp").alias("customer_ingestion_timestamp")
    )

ref_events_customers.write \
    .mode("overwrite") \
    .parquet(f"{refined_path}/ref_events_customers")

ref_events_customers.show()
ref_events_customers.printSchema()

+-----------+--------+----------+-------------------+-------------------------+-----------+---------------+-------+----------------------------+
|customer_id|event_id|event_type|    event_timestamp|event_ingestion_timestamp|       name|          email|country|customer_ingestion_timestamp|
+-----------+--------+----------+-------------------+-------------------------+-----------+---------------+-------+----------------------------+
|          1|      E1|     login|2023-07-10 10:01:00|     2026-04-16 00:46:...|  Ana Silva|  ana@email.com|     BR|        2026-04-16 00:46:...|
|          1|      E2|  purchase|2023-07-10 10:05:00|     2026-04-16 00:46:...|  Ana Silva|  ana@email.com|     BR|        2026-04-16 00:46:...|
|          2|      E3|     login|2023-07-11 09:00:00|     2026-04-16 00:46:...|   John Doe| john@email.com|     US|        2026-04-16 00:46:...|
|         99|      E4|     login|2023-07-11 09:10:00|     2026-04-16 00:46:...|       NULL|           NULL|   NULL|               

### 📊 Análises Ad Hoc

Os próximos blocos apresentam análises exploratórias (ad hoc) realizadas sobre os dados tratados.

Essas análises têm como objetivo extrair insights pontuais de negócio, validar hipóteses e demonstrar o potencial analítico das bases, sem a necessidade de modelagem formal ou estruturação definitiva.

São consultas flexíveis, direcionadas por perguntas específicas, e podem evoluir posteriormente para métricas padronizadas ou dashboards.


In [43]:
# quantidade de clientes
df_customers_trusted.select("customer_id").distinct().count()

5

In [54]:
# receita de pedidos concluidos
from pyspark.sql.functions import col, sum as _sum

df_orders_enriched \
    .filter(col("status") == "completed") \
    .select(_sum("amount_brl").alias("total_revenue_brl")) \
    .show()

+-----------------+
|total_revenue_brl|
+-----------------+
|          2866.38|
+-----------------+



In [45]:
# quantos clientes fizeram login
df_events_trusted \
    .filter(col("event_type") == "login") \
    .count()

5

In [46]:
# ticket médio
df_orders_enriched \
    .groupBy("customer_id") \
    .agg(_sum("amount_brl").alias("total_spent")) \
    .selectExpr("avg(total_spent) as avg_ticket") \
    .show()

+----------+
|avg_ticket|
+----------+
|   673.076|
+----------+



In [48]:
# faturamento por país
df_orders_enriched \
    .join(df_customers_trusted, "customer_id") \
    .groupBy("country") \
    .agg(_sum("amount_brl").alias("total_revenue")) \
    .orderBy(col("total_revenue").desc()) \
    .show()

+-------+-------------+
|country|total_revenue|
+-------+-------------+
|     US|        499.0|
|     BR|        200.5|
+-------+-------------+



In [56]:
# faturamento por mês
from pyspark.sql.functions import month, year, sum as _sum

df_orders_enriched \
    .filter(col("status") == "completed") \
    .withColumn("year", year("order_date")) \
    .withColumn("month", month("order_date")) \
    .groupBy("year", "month") \
    .agg(_sum("amount_brl").alias("revenue")) \
    .orderBy("year", "month") \
    .show()

+----+-----+-------+
|year|month|revenue|
+----+-----+-------+
|2023|    7|2866.38|
+----+-----+-------+

